# FRTB Standardised Approach — Capital Charge Calculator
**Scope:** Equity Risk — Delta Sensitivity Based Method  
**Portfolio:** 5-Stock NSE Equity Book  
**Author:** Mahek | FRM  
**Reference:** BCBS FRTB SA Framework

## 1. Portfolio Construction

In [3]:
import pandas as pd
import numpy as np

#Portfolio positions

portfolio = pd.DataFrame({
    'stock' : ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK' ],
    'sector' : ['Energy', 'IT', 'Financials', 'IT', 'Financials'],
    'market_value' : [2000000, 1500000,1000000,1500000,1000000],
    'price': [2850, 3900, 1650, 1450, 1100]
})

#Delta Sensitivity = market value * 1%

portfolio['delta_sensitivity'] = portfolio['market_value'] * 0.01

print(portfolio)
print(f"\n Total Portfolio value: ${portfolio['market_value'].sum():,.0f}")


       stock      sector  market_value  price  delta_sensitivity
0   RELIANCE      Energy       2000000   2850            20000.0
1        TCS          IT       1500000   3900            15000.0
2   HDFCBANK  Financials       1000000   1650            10000.0
3       INFY          IT       1500000   1450            15000.0
4  ICICIBANK  Financials       1000000   1100            10000.0

 Total Portfolio value: $7,000,000


## 2. Delta Sensitivities & Risk Weights 

In [4]:
#FRTB SA Equity Risk Weights by Sector

risk_weights = {
    'Energy' : 0.55,
    'IT' : 0.60,
    'Financials' : 0.70
}

#Apply Risk Weight by Position

portfolio['risk_weight'] = portfolio['sector'].map(risk_weights)
portfolio['weighted_sensitivity'] = portfolio['delta_sensitivity'] * portfolio['risk_weight']

print(portfolio[['stock', 'sector', 'delta_sensitivity', 'risk_weight', 'weighted_sensitivity']])

       stock      sector  delta_sensitivity  risk_weight  weighted_sensitivity
0   RELIANCE      Energy            20000.0         0.55               11000.0
1        TCS          IT            15000.0         0.60                9000.0
2   HDFCBANK  Financials            10000.0         0.70                7000.0
3       INFY          IT            15000.0         0.60                9000.0
4  ICICIBANK  Financials            10000.0         0.70                7000.0


## 3. Intra-Bucket Aggregation

In [ ]:
correlations = {
    'high': 0.75,
    'medium': 0.50,
    'low': 0.25
}

def bucket_capital_charge(sensitivities, correlation):
    n = len(sensitivities)
    sum_squared = np.sum(sensitivities**2)
    cross_terms = 0
    for i in range(n):
        for j in range(n):
            if i!=j:
                cross_terms += correlation * sensitivities[i] * sensitivities[j]
    return np.sqrt(sum_squared + cross_terms)
            
results = []

for scenario, rho in correlations.items():
    for sector in portfolio['sector'].unique():
        bucket = portfolio[portfolio['sector'] == sector]['weighted_sensitivity'].values
        kb = bucket_capital_charge(bucket, rho)
        results.append({
            'scenario' : scenario,
            'sector' : sector,
            'kb' : kb
        })
results_df = pd.DataFrame(results)
print(results_df.sort_values(['scenario', 'sector']))

  scenario      sector            kb
0     high      Energy  11000.000000
2     high  Financials  13095.800854
1     high          IT  16837.458240
6      low      Energy  11000.000000
8      low  Financials  11067.971811
7      low          IT  14230.249471
3   medium      Energy  11000.000000
5   medium  Financials  12124.355653
4   medium          IT  15588.457268


## 4. Inter-Bucket Aggregation

In [17]:
gamma = 0.15

total_charges = []

for scenario in ['high', 'medium', 'low']:
    bucket_charges = results_df[results_df['scenario'] == scenario]['kb'].values

    #Inter_bucket Aggregation
    sum_squared = np.sum(bucket_charges ** 2)
    cross_terms = 0
    n = len(bucket_charges)

    for i in range(n):
        for j in range(n):
            if i != j:
                cross_terms += gamma*bucket_charges[i] * bucket_charges[j]

    total_kb = np.sqrt(sum_squared + cross_terms)
    total_charges.append({
        'scenario' : scenario,
        'total_capital_charge' : total_kb
    })

    final = pd.DataFrame(total_charges)
    print("\n=== FRTB SA CAP CHG ===")
    print(final)
    print(f"\n Worst Case Capital Charge: ${final['total_capital_charge'].max():,.0f}")


=== FRTB SA CAP CHG ===
  scenario  total_capital_charge
0     high           27220.02489

 Worst Case Capital Charge: $27,220

=== FRTB SA CAP CHG ===
  scenario  total_capital_charge
0     high          27220.024890
1   medium          25673.961179

 Worst Case Capital Charge: $27,220

=== FRTB SA CAP CHG ===
  scenario  total_capital_charge
0     high          27220.024890
1   medium          25673.961179
2      low          24015.289510

 Worst Case Capital Charge: $27,220


## 5. Capital Charge Summary

In [18]:
print("=" * 55)
print(f"{'FRTB SA CAPITAL CHARGE SUMMARY':^55}")
print(f"{'Portfolio Value: ₹70,00,000':^55}")
print("=" * 55)
print(f"\n{'Intra-bucket results':}")
print("-" * 55)
print(f"{'Scenario':<15} {'Energy':>12} {'IT':>12} {'Financials':>12}")
print("-" * 55)

for scenario in ['high', 'medium', 'low']:
    row = results_df[results_df['scenario'] == scenario]
    energy = row[row['sector'] == 'Energy']['kb'].values[0]
    it = row[row['sector'] == 'IT']['kb'].values[0]
    fin = row[row['sector'] == 'Financials']['kb'].values[0]
    print(f"{scenario:<15} ₹{energy:>10,.0f} ₹{it:>10,.0f} ₹{fin:>10,.0f}")

print("=" * 55)
print(f"\n{'Total Capital Charge by Scenario':}")
print("-" * 55)
for _, row in final.iterrows():
    print(f"{row['scenario']:<15} ₹{row['total_capital_charge']:>10,.0f}")

print("=" * 55)
print(f"{'WORST CASE':^55}")
print(f"₹{final['total_capital_charge'].max():^53,.0f}")
print("=" * 55)

            FRTB SA CAPITAL CHARGE SUMMARY             
              Portfolio Value: ₹70,00,000              

Intra-bucket results
-------------------------------------------------------
Scenario              Energy           IT   Financials
-------------------------------------------------------
high            ₹    11,000 ₹    16,837 ₹    13,096
medium          ₹    11,000 ₹    15,588 ₹    12,124
low             ₹    11,000 ₹    14,230 ₹    11,068

Total Capital Charge by Scenario
-------------------------------------------------------
high            ₹    27,220
medium          ₹    25,674
low             ₹    24,015
                      WORST CASE                       
₹                       27,220                        
